# Data Preprocessing & Feature Engineering — Churn Prediction

### 📌 Recap

From [01 Project Introduction.ipynb](01%20Project%20Introduction.ipynb): predict bank customer churn (`Exited`) with an ANN. Before any model can be built, the raw `Churn_Modelling.csv` needs cleaning: drop non-predictive columns, encode categorical features into numbers, split into train/test, and scale the features. Every transformer (encoders, scaler) gets **saved as a pickle file**, since the exact same transformations will need to be replayed on new user input at deployment time.

This is the same feature engineering process used throughout the earlier machine learning material — nothing new conceptually, just applied here as the first stage of the ANN pipeline.

## 1. Load the dataset

In [25]:
import pandas as pd

data = pd.read_csv("../Churn_Modelling.csv")
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 2. Drop irrelevant (non-predictive) columns

`RowNumber`, `CustomerId`, and `Surname` are identifiers — they don't carry any signal about whether a customer will churn, so they're dropped before any modeling.

In [26]:
data = data.drop(["RowNumber", "CustomerId", "Surname"], axis=1)
data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 3. Encode categorical features

`Gender` and `Geography` are both categorical (text), but a neural network only accepts numbers — so both need encoding. They're handled **differently**, and the reason why is an important ML fundamentals point:

- **`Gender`** has exactly **2** categories (`Male`/`Female`) → **Label Encoding** is fine here: mapping to `0`/`1` doesn't imply any false ordering, since there's no meaningful "greater than" relationship being introduced between just two categories.
- **`Geography`** has **3** categories (`France`/`Spain`/`Germany`) → Label Encoding would be a **mistake** here. Mapping them to e.g. `France=0, Spain=1, Germany=2` would make the model treat `Germany > Spain > France` numerically, implying a false ordinal relationship between countries that doesn't actually exist. **One-Hot Encoding** avoids this by giving each category its own binary column instead.

### 3a. `Gender` → Label Encoding

In [27]:
from sklearn.preprocessing import LabelEncoder

label_encoder_gender = LabelEncoder()
data["Gender"] = label_encoder_gender.fit_transform(data["Gender"])

print(dict(zip(label_encoder_gender.classes_, label_encoder_gender.transform(label_encoder_gender.classes_))))
data.head()

{'Female': 0, 'Male': 1}


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


### 3b. `Geography` → One-Hot Encoding

> ⚠️ **Two version-related gotchas the video ran into (already fixed here, but worth knowing why):**
> 1. **`OneHotEncoder(sparse=False)` no longer works** on recent scikit-learn (this environment has 1.9.0) — the `sparse` parameter was renamed to **`sparse_output`** and the old name was fully removed. Use `OneHotEncoder(sparse_output=False)` instead.
> 2. If you *do* leave the default `sparse_output=True`, the result is a SciPy sparse matrix — passing that directly into `pd.DataFrame(sparse_matrix, columns=[...])` doesn't expand it into separate columns; pandas treats it as one opaque object column, causing a `(10000, 1)` vs `(10000, 3)` shape mismatch. Calling `.toarray()` first (or just setting `sparse_output=False` up front, as done below) avoids this entirely.

In [28]:
from sklearn.preprocessing import OneHotEncoder

one_hot_encoder_geo = OneHotEncoder(sparse_output=False)
geo_encoded = one_hot_encoder_geo.fit_transform(data[["Geography"]])

geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=one_hot_encoder_geo.get_feature_names_out(["Geography"]),
)
geo_encoded_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


### 3c. Combine the encoded columns back into the main dataframe

Drop the original `Geography` text column (no longer needed) and concatenate the new one-hot columns in its place.

In [29]:
data = pd.concat([data.drop("Geography", axis=1), geo_encoded_df], axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


## 4. Save the encoders as pickle files

These exact fitted encoders (not just the *logic*, but the specific category→number mappings they learned) will be needed again at deployment time, to transform new raw user input the same way the training data was transformed. Saving them now means they don't need to be refit later.

In [30]:
import pickle

with open("label_encoder_gender.pkl", "wb") as file:
    pickle.dump(label_encoder_gender, file)

with open("onehot_encoder_geo.pkl", "wb") as file:
    pickle.dump(one_hot_encoder_geo, file)

print("Saved: label_encoder_gender.pkl, onehot_encoder_geo.pkl")

Saved: label_encoder_gender.pkl, onehot_encoder_geo.pkl


## 5. Split into independent (`X`) and dependent (`y`) features

`Exited` is the target; everything else is now a numeric, model-ready feature.

In [31]:
X = data.drop("Exited", axis=1)
y = data["Exited"]

print("X shape:", X.shape, " y shape:", y.shape)
X.columns.tolist()

X shape: (10000, 12)  y shape: (10000,)


['CreditScore',
 'Gender',
 'Age',
 'Tenure',
 'Balance',
 'NumOfProducts',
 'HasCrCard',
 'IsActiveMember',
 'EstimatedSalary',
 'Geography_France',
 'Geography_Germany',
 'Geography_Spain']

`X`'s column count should match the "11 input nodes" figure estimated in the [project introduction notebook](01%20Project%20Introduction.ipynb#Why-11-input-features) — but that was an *estimate* made before this code existed, assuming `OneHotEncoder(drop='first')`. Let's check the actual count produced by this notebook's real code, rather than assume the estimate was exact.

In [32]:
print(f"Actual feature count: {X.shape[1]}")
print(X.columns.tolist())

Actual feature count: 12
['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_France', 'Geography_Germany', 'Geography_Spain']


> 📝 **Note:** this comes out to **12** features, not 11 — because `OneHotEncoder` here keeps **all 3** Geography categories as separate columns (`France`, `Germany`, `Spain`) rather than dropping one to avoid redundancy (which would need `OneHotEncoder(drop="first")`). The project introduction notebook's "11 features" estimate assumed a dropped first category; this notebook's actual code keeps all 3, giving 12. Neither is wrong — keeping all one-hot columns is simpler and perfectly valid for a neural network (unlike linear models, NNs don't suffer from the "dummy variable trap" the same way) — just noting the discrepancy from the earlier estimate rather than leaving it unverified.

## 6. Train / test split

In [33]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train:", X_train.shape, " X_test:", X_test.shape)

X_train: (8000, 12)  X_test: (2000, 12)


## 7. Feature scaling — `StandardScaler`

Neural networks train much better when input features are on a similar scale (e.g. `Balance` can be in the hundreds of thousands, while `HasCrCard` is just `0`/`1`) — large unscaled magnitudes can dominate the loss landscape and slow down or destabilize training.

> 💡 **Fit only on the training set, then transform both.** `scaler.fit_transform(X_train)` learns the mean/standard deviation **from training data only**, then `scaler.transform(X_test)` applies those *same* learned statistics to the test set. Fitting on the test set too (or on the combined data) would leak information from the test set into preprocessing — a subtle form of data leakage that inflates evaluation results. The video correctly follows this pattern.

In [34]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train[:3]

array([[ 0.35649971,  0.91324755, -0.6557859 ,  0.34567966, -1.21847056,
         0.80843615,  0.64920267,  0.97481699,  1.36766974,  1.00150113,
        -0.57946723, -0.57638802],
       [-0.20389777,  0.91324755,  0.29493847, -0.3483691 ,  0.69683765,
         0.80843615,  0.64920267,  0.97481699,  1.6612541 , -0.99850112,
         1.72572313, -0.57638802],
       [-0.96147213,  0.91324755, -1.41636539, -0.69539349,  0.61862909,
        -0.91668767,  0.64920267, -1.02583358, -0.25280688, -0.99850112,
        -0.57946723,  1.73494238]])

> 📝 **Worth knowing:** this scales *every* column in `X_train`/`X_test`, including the `0`/`1` one-hot and label-encoded columns (`Gender`, `Geography_*`), not just the continuous ones (`Balance`, `Age`, ...). This is a common, generally-harmless simplification for neural network inputs — scaling binary columns doesn't distort their meaning (they just become two symmetric values instead of `0`/`1`) — but it's a deliberate simplification worth being aware of, not an oversight.

## 8. Save the scaler as a pickle file

In [35]:
with open("scaler.pkl", "wb") as file:
    pickle.dump(scaler, file)

print("Saved: scaler.pkl")

Saved: scaler.pkl


## 9. Summary

- Dropped 3 non-predictive identifier columns (`RowNumber`, `CustomerId`, `Surname`).
- Encoded `Gender` with **Label Encoding** (safe — only 2 categories, no false ordering introduced) and `Geography` with **One-Hot Encoding** (necessary — 3 categories, avoids implying a false rank order between countries).
- Saved both fitted encoders (`label_encoder_gender.pkl`, `onehot_encoder_geo.pkl`) for reuse at deployment time.
- Split into `X` (12 features after encoding) / `y` (`Exited`), then into train/test sets (80/20).
- Scaled features with `StandardScaler`, fit on **training data only** to avoid data leakage, and saved it (`scaler.pkl`).
- **Corrections applied:** `OneHotEncoder`'s `sparse` parameter → `sparse_output` (renamed in recent scikit-learn); noted the sparse-matrix-to-DataFrame conversion gotcha; corrected the "11 features" estimate from notebook 01 to the actual **12** (since all 3 Geography categories are kept, not dropped to 2).

Three pickle files should now exist alongside this notebook: `label_encoder_gender.pkl`, `onehot_encoder_geo.pkl`, `scaler.pkl`.

## 10. Likely exam / interview questions

1. Why is Label Encoding safe for `Gender` but not for `Geography`?
2. What problem does One-Hot Encoding solve that Label Encoding doesn't, for a multi-category feature?
3. Why must `StandardScaler` be fit only on the training set, not the test set?
4. Why are the encoders and scaler saved as pickle files, rather than just applying the logic again later?
5. What causes a SciPy sparse matrix to produce a `(n, 1)` shaped column instead of expanding into separate columns when passed to `pd.DataFrame`?
6. What is the "dummy variable trap", and why doesn't it matter much for neural networks the way it does for linear models?

## 11. What's next

- Build and train the ANN itself with Keras/TensorFlow: architecture, activation functions, loss, optimizer, dropout, and TensorBoard logging.